# Logistic Regression: Predicting Earthquake Damage

## Introduction

In Lesson 1, we queried the Nepal earthquake database and created the `duckdb_wrangle` module that loads clean, model-ready data with a single function call. We also defined our binary target: `severe_damage = 1` for Grade 4 or 5 damage, `0` otherwise.

Today we build our **first classification model** — logistic regression — to predict whether a building in Gorkha district will sustain severe damage based on its physical characteristics.

> ❓ **The core question:** can a building's physical features (age, foundation type, roof material, height) reliably predict whether it suffered severe damage in the 2015 earthquake? If yes, this model could help prioritize buildings for inspection before the next earthquake — and help identify construction patterns to avoid in future building.

This lesson introduces the full machine learning workflow end-to-end: load data, explore it, split it, encode features, train a model, evaluate it with multiple metrics, and interpret what the model learned.

## Learning Objectives

By the end of this lesson, you will be able to:

1. Distinguish **classification** from **regression** and explain why earthquake damage prediction is a binary classification task
2. Explain how **logistic regression** converts a linear combination of features into a probability via the **sigmoid function**
3. Implement the standard **train/test split** workflow and explain why it prevents overfitting
4. Define **data leakage** and identify at least two specific ways it can corrupt a model evaluation
5. Apply **One-Hot Encoding** (OHE) via a sklearn `Pipeline` that prevents leakage
6. Compute and interpret **accuracy, precision, recall, confusion matrix**, and **ROC/AUC**
7. Compute **odds ratios** from logistic regression coefficients and explain what they tell you about feature importance


## Part 1: Classification vs Regression

Supervised learning comes in two fundamental flavors. The difference lies in what the model predicts.

### Regression: Predicting Continuous Values

**Regression** predicts a **continuous number** — any value across a range. Examples:

- House price: \$150,000, \$275,500, \$1,200,000 (infinitely many possibilities on a continuous number line)
- Temperature tomorrow: 72.3°F, 68.7°F (any decimal value)
- Earthquake magnitude: 5.2, 6.8, 7.1

The output can take any real value between negative infinity and positive infinity (or some bounded range). The model is wrong by a *degree* — "off by USD 20,000" is meaningful.

### Classification: Predicting Categories

**Classification** predicts a **discrete category** — one of a fixed set of choices. Examples:

- Email: Spam or Not Spam (two categories)
- Building damage: Severe or Not Severe (two categories)
- Handwritten digit: 0, 1, 2, …, 9 (ten categories)
- Disease diagnosis: Cancer or Not Cancer (two categories)

The output is a category label. The model is either right or wrong — there is no "off by a little."

**Binary classification** is classification with exactly **two classes**. Our task is binary: every building in Gorkha is either **severely damaged (1) or not (0)**.

| | Regression | Classification |
|---|---|---|
| Output | Continuous number | Discrete category |
| Error type | Magnitude (off by X units) | Correct / Incorrect |
| Example loss | Mean Squared Error | Cross-entropy |
| Example metric | R², MAE, RMSE | Accuracy, Precision, Recall |
| Nepal P4 usage | Not used in P4 | This lesson onwards |

> 🧠 **A common misconception:** logistic regression has "regression" in its name but is used for *classification*. The "regression" refers to the internal linear combination (`z = β₀ + β₁x₁ + ...`) — the output is then converted to a probability and thresholded into a class label.


### The Nepal Context

In Lesson 1, we created the binary target:

```
severe_damage = 1  if Grade 4 or Grade 5 (collapsed or unusable)
severe_damage = 0  if Grade 1, 2, or 3 (intact, minor, or moderate damage)
```

We are not predicting "how much damage?" (regression). We are predicting "did this building sustain severe damage? Yes or No?" (binary classification).

**Why binary instead of 5-class ordinal?** Three reasons:
1. **Practical decision**: disaster response teams face a binary choice — does this building need immediate major intervention or not?
2. **Modeling simplicity**: binary classification is easier to evaluate (clear precision/recall semantics) and the model has a simple output probability
3. **Class balance**: collapsing to binary produces ~64% severe vs ~36% not-severe, which is manageable

### Decision Thresholds

A classification model does not output 0 or 1 directly. Instead, it outputs a **probability** — a number between 0 and 1.

The model says: *"I believe this building has a 0.72 probability of severe damage."*

To convert that probability into a **class label** (0 or 1), we apply a **decision threshold** (default: 0.5):

```
If P(severe) < 0.5  →  Predict: Not severe (0)
If P(severe) ≥ 0.5  →  Predict: Severe (1)
```

| P(severe) | Prediction     | Certainty |
|-----------|----------------|-----------|
| 0.10      | 0 (not severe) | High confidence — safe |
| 0.48      | 0 (not severe) | Low confidence — borderline |
| 0.50      | 1 (severe)     | Boundary case |
| 0.72      | 1 (severe)     | Moderate confidence |
| 0.95      | 1 (severe)     | High confidence — almost certainly damaged |

> ⚠️ **The threshold is adjustable.** In disaster response, we might lower the threshold to 0.4 — meaning we flag more buildings as "possibly severe" even if the model is only 40% confident. This reduces false negatives (missed damaged buildings) at the cost of more false positives (extra inspections of safe buildings). We explore this trade-off explicitly when we look at the ROC curve in Part 12.


## Part 2: Logistic Regression and the Sigmoid Function

Logistic Regression is a **linear model for classification**. It transforms a linear combination of features into a probability.

### Step 1: The Linear Combination

Start with a weighted sum of features — the same form as linear regression:

```
z = β₀ + β₁·(age) + β₂·(plinth_area) + β₃·(height) + β₄·(foundation_RC) + ...
```

Each β (beta) is a learned coefficient. For a building with age=15, plinth_area=120, height=18, and RC foundation:

```
z = -2.1 + 0.05·(15) + 0.002·(120) + 0.08·(18) + (-1.3)·(1)
  = -2.1 + 0.75 + 0.24 + 1.44 + (-1.30)
  = -0.97
```

At this point, `z` is just a number on the real line. It could be:
- **Negative** (strong evidence against severe damage — RC foundations reduce damage odds)
- **Zero** (model is uncertain)
- **Positive** (strong evidence for severe damage)

But `z` is not yet a probability — it can range from −∞ to +∞. We need to squash it into [0, 1].

### Why Can't We Use Linear Regression Directly?

Linear regression would predict `P(severe) = β₀ + β₁x₁ + ...`. For extreme feature values, this can produce predictions below 0 or above 1 — which are not valid probabilities. Logistic regression solves this by applying the sigmoid function.

> 💡 **What is being learned?** During training, scikit-learn's `LogisticRegression` finds the β values that maximize the likelihood of the observed training labels. This is done through numerical optimization (gradient descent or L-BFGS), not a closed-form solution like linear regression.


### Step 2: The Sigmoid Function

To convert `z` into a **probability** between 0 and 1, we apply the **sigmoid function**:

$$P(\text{severe}) = \frac{1}{1 + e^{-z}}$$

where `e ≈ 2.718` (Euler's number / the base of the natural logarithm).

**Why does this work?**
- When `z → +∞`: `e^{-z} → 0`, so `P → 1/(1+0) = 1.0`
- When `z = 0`: `e^0 = 1`, so `P = 1/(1+1) = 0.5`
- When `z → -∞`: `e^{-z} → +∞`, so `P → 1/(1+∞) = 0.0`

**The sigmoid produces the characteristic S-shaped curve:**

```
P(severe) │
      1.0 ┤                           ╭──────────────
      0.9 ┤                       ╭───╯
      0.8 ┤                   ╭───╯
      0.7 ┤               ╭───╯
      0.6 ┤           ╭───╯
      0.5 ┼───────────╮─────────  ← Default decision threshold
      0.4 ┤       ╭───╯
      0.3 ┤   ╭───╯
      0.2 ┤╭──╯
      0.1 ┤╯
      0.0 ┤────────────────────────
        │ -4 -2  0  2  4
        └─────────────────────── z (linear combination)
```

| z value | P(severe) | Interpretation |
|---------|-----------|----------------|
| −4.0 | 0.018 | Strong evidence: NOT severe |
| −2.0 | 0.119 | Moderate evidence: NOT severe |
|  0.0 | 0.500 | Uncertain (decision boundary) |
| +2.0 | 0.881 | Moderate evidence: SEVERE |
| +4.0 | 0.982 | Strong evidence: SEVERE |

> 📌 **The sigmoid is monotonically increasing** — as `z` increases, `P` always increases. This means the model never "flips" its prediction as a feature increases continuously.

> 🧠 **The name "logistic regression" comes from the logistic function**, another name for the sigmoid. The model is linear in the log-odds (logit) space:
> `log(P / (1-P)) = z = β₀ + β₁x₁ + ...`
> This log-odds framing is what makes **odds ratios** interpretable (Part 13).


### Step 3: The Decision Boundary

At **P = 0.5**, the linear combination is exactly **z = 0**. This is the **decision boundary** — the surface in feature space that separates predicted-severe from predicted-not-severe.

```
z < 0  →  P < 0.5  →  Predict NOT severe
z = 0  →  P = 0.5  →  Exactly on the boundary
z > 0  →  P > 0.5  →  Predict SEVERE
```

**Concrete Nepal examples:**

- **Mud mortar foundation, old building**: β·x is positive → z is positive → P(severe) > 0.5 → predicted SEVERE
- **RC engineered foundation, newer building**: β·x for RC is strongly negative (RC resists damage) → z is negative → P(severe) < 0.5 → predicted NOT SEVERE

> 💡 **The decision boundary is a hyperplane** (a line in 2D, a plane in 3D, a hyperplane in high-dimensional space). Logistic Regression can only separate classes with a straight-line boundary. If the true boundary is curved (e.g., "buildings with moderate age AND stone foundations are damaged, but very old or very new buildings of the same type are not"), logistic regression may underfit. Decision Trees (Lesson 3) can learn curved boundaries.

### What Do the Probabilities Mean?

Logistic Regression gives you **calibrated probabilities** — meaning a prediction of P = 0.70 implies that among all buildings with that feature profile, approximately 70% actually sustained severe damage. This calibration property makes the model useful for risk scoring, not just binary classification.

> ➡️ Now that we understand the mathematical form of the model, let's look at the full machine learning workflow — where logistic regression fits in the pipeline.


## Part 3: The Machine Learning Workflow

Every machine learning project follows a standard pipeline. Understanding this pipeline helps you see where we are in the process and what comes next.

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│  LOAD    │──▶│ EXPLORE  │──▶│  SPLIT   │──▶│  ENCODE  │
│  Data    │   │  Data    │   │ Train/   │   │ Features │
│  (L1)    │   │  (L2)    │   │  Test    │   │  (L2)    │
└──────────┘   └──────────┘   └──────────┘   └────┬─────┘
                                                   │
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌────▼─────┐
│ INTERPRET│◀──│EVALUATE  │◀──│ PREDICT  │◀──│  TRAIN   │
│  Model   │   │ Metrics  │   │ on Test  │   │  Model   │
│  (L2)    │   │  (L2)    │   │  (L2)    │   │  (L2)    │
└──────────┘   └──────────┘   └──────────┘   └──────────┘
```

> 📌 **This 8-step pipeline is the backbone of every supervised learning project in this course.** L3 (Decision Trees), L4 (Demographics), and L5 (Assignment) all follow the same template. Learn it deeply in L2, and the rest of the course becomes a set of variations on this theme.


### The Eight Steps

| Step | What happens | Where in this lesson |
|------|-------------|---------------------|
| **1. Load Data** | Read from database or CSV via wrangle function | Part 7 (Code Task 4.2.1.1) |
| **2. Explore** | Visualize distributions, correlations, class balance | Part 8 (Code Tasks 4.2.2.1–4.2.2.4) |
| **3. Split** | Divide into 80% training, 20% test — **before** any transformations | Part 9 (Code Task 4.2.3.3) |
| **4. Encode** | Convert categorical text → numeric binary columns (OHE) | Part 11, inside Pipeline |
| **5. Train** | Fit logistic regression on training data only | Part 11 (Code Task 4.2.5.1) |
| **6. Predict** | Apply model to test data → predictions | Part 12 (Code Task 4.2.6.1) |
| **7. Evaluate** | Accuracy, precision, recall, confusion matrix, ROC/AUC | Part 12 (Code Tasks 4.2.6.1–4.2.6.2) |
| **8. Interpret** | Coefficients → odds ratios → feature importance | Part 13 (Code Task 4.2.7.1) |

> ⚠️ **Step 3 must come before Step 4.** Fitting any transformer (encoder, scaler) on the full dataset before splitting causes data leakage — covered in detail in Part 4. The sklearn Pipeline enforces the correct order automatically.

### Lesson Progression in Context

| Lesson | Primary topic | New techniques added |
|--------|--------------|---------------------|
| L1 | SQL, databases, wrangle | SQLite, DuckDB, `wrangle_nepal_data()` |
| **L2 (this)** | **Logistic Regression** | OHE, Pipeline, accuracy, precision, recall, ROC/AUC, odds ratios |
| L3 | Decision Trees | `max_depth`, Gini impurity, feature importance, bias-variance |
| L4 | Demographics + JOINs | Multi-table SQL, equity analysis, caste × damage |
| L5 | End-to-end assignment | Full pipeline built from scratch |


## Part 4: Train/Test Split and Data Leakage

Before we build our first model, we need to understand one of the **most critical principles** in machine learning: why we must split our data, and what can go catastrophically wrong.

### Why Do We Need to Generalize?

A machine learning model is only useful if it works on **new, unseen data**. A model trained on 2015 Gorkha earthquake data must predict damage for the next earthquake in a different district. A bank's fraud detector trained on 2023 data must catch fraud in 2025.

**If a model only works on the data it was trained on, it is useless in practice.**

### Training Accuracy is Misleading

Imagine a student who memorizes the answers to every practice exam problem. On practice exams, they score 100%. On the final exam with new problems, they score 35%. **Memorization is not understanding.**

The same happens with ML models. A model can achieve near-100% accuracy on training data by memorizing patterns specific to that data — patterns that vanish on new buildings.

```
Training Data Accuracy: 98%  ← Model memorized training set
Test Data Accuracy:     71%  ← Real performance on new data
```

This gap is called **overfitting**: the model has fit the training data so well that it has captured noise and idiosyncrasies, not the true underlying pattern.

> 🔍 **How to diagnose overfitting:** compare training accuracy to test accuracy. A large gap (e.g., 98% train vs 71% test) signals overfitting. In L3, we will use the `max_depth` parameter of Decision Trees to control overfitting directly.


### The Train/Test Split

We prevent overfitting by dividing the data into two groups **before** any model training:

```
Full Dataset (70,836 buildings in Gorkha)
├── Training Set (80% = ~56,668 buildings)  ──▶  Use to train model
└── Test Set     (20% = ~14,168 buildings)  ──▶  Use to evaluate model
                                                  (never seen during training)
```

The model learns patterns from the **training set only**. We then measure its performance on the **test set**, which it has never seen — simulating how the model would perform on new data in the real world.

**Why 80/20?** It's a common convention that balances:
- **Enough training data** to learn good patterns
- **Enough test data** for reliable evaluation (too few test examples → high variance in metrics)

Alternatives like 70/30 or 90/10 are also used depending on dataset size. With 70,836 buildings, 80/20 gives ~14,000 test examples — more than enough for stable evaluation.

**`random_state=42`**: Setting the random seed makes the split reproducible. Every run of the code will produce the same train/test assignment, so results can be compared across experiments.

> ⚠️ **The critical rule:** once you define your test set, **never train on it and never use test labels to make any decisions**. Treat it like a sealed exam — you can only open it for final evaluation.


### Data Leakage — The Silent Killer of Production Models

**Data leakage** occurs when information from the test set "leaks" into the training process — meaning the model inadvertently sees or is influenced by data it should not have access to during training.

**Formal definition:** Data leakage is present when information that would **not be available at prediction time** is used during training.

> ⚠️ **Why "silent killer"?** Leakage artificially inflates evaluation metrics, making a flawed model look production-ready. The model is not learning to predict from building features — it's learning to cheat using information it won't have in the real world. Deploy that model and it fails catastrophically on real earthquakes.

---

> ⚠️ **Leakage Scenario 1: Fitting the encoder on the full dataset before splitting**
>
> This is the most common mistake when using categorical encoders.
>
> ```python
> # ❌ WRONG — encoder sees test data during fit
> X_full = df[features]
> encoder = OneHotEncoder()
> encoder.fit(X_full)                      # LEAKS: test set influences encoder
> X_encoded = encoder.transform(X_full)
>
> # Now split the already-encoded data
> X_train, X_test = X_encoded[:n_train], X_encoded[n_train:]
> ```
>
> **What goes wrong:**
> - The encoder learned the category distribution from the full dataset (including test set)
> - In deployment, if an unknown category appears, the encoder fails silently
> - The test set's category distribution influences how training examples are encoded
> - Evaluation metrics are artificially inflated
>
> **The fix:** use a Pipeline that fits the encoder ONLY on training data:
> ```python
> # ✓ CORRECT — Pipeline ensures encoder is fit only on X_train
> model = Pipeline([('encoder', OneHotEncoder()), ('clf', LogisticRegression())])
> model.fit(X_train, y_train)  # encoder fits on X_train only
> model.predict(X_test)        # encoder transforms X_test using train-fit params
> ```


> ⚠️ **Leakage Scenario 2: Using post-event features as predictors**
>
> In a disaster-prediction context, some features are only available *after* the earthquake occurs. Including them is leakage because in real deployment, the model would be asked to predict damage *before* knowing what happened.
>
> ```python
> # ❌ WRONG — reconstruction_cost only known AFTER damage is assessed
> features = ['age_building', 'height_ft_pre_eq', 'reconstruction_cost']
>              ↑ pre-event          ↑ pre-event       ↑ POST-EVENT leakage
> ```
>
> **Nepal-specific examples of post-event leakage:**
>
> | Feature | Type | Why it leaks |
> |---------|------|-------------|
> | `condition_post_eq` | Post-event | Directly describes damage state — trivially predicts the target |
> | `reconstruction_cost` | Post-event | Only estimated after damage is assessed |
> | `technical_solution_proposed` | Post-event | Proposed based on observed damage |
> | `damage_grade` (raw) | Post-event | IS the target in disguised form |
>
> Our `wrangle_nepal_data()` function removes `damage_grade` from features precisely to prevent this. But `condition_post_eq` could also be a leakage risk — it describes the building's post-earthquake state, which is correlated with but not identical to `damage_grade`. In our feature set, we exclude it.
>
> **The rule:** features must represent information available at the time you would use the model in production. Ask: "Would I know this before the earthquake?"


### What Leakage Does to Metrics

| Without leakage | With Scenario 1 leakage | With Scenario 2 leakage |
|----------------|------------------------|------------------------|
| 72% accuracy | 85-90% (inflated) | 95%+ (severely inflated) |
| Honest evaluation | Optimistic evaluation | Almost meaningless evaluation |
| Real-world deployable | May fail in deployment | Will fail in deployment |

> 🧠 **The danger of looking too good:** a model that achieves 95% accuracy on a classification task where the base rate is 64% should always be scrutinized. Ask: "Is there a source of leakage here?" Genuinely 95% accuracy on earthquake damage prediction from pre-event building features alone would be extraordinary — extraordinary claims require scrutiny.

### Why Test Sets Must Stay Untouched

Once the test set is sealed, **never**:
- Peek at test labels to tune the model (this is **test set leakage**)
- Refit any transformer after seeing test data
- Report metrics on the training set as if they represent generalization performance
- Add samples to the test set from the same time period as training

In sklearn, the **Pipeline** pattern automatically prevents the most common leakage: it fits all transformers (encoders, scalers) on training data only during `model.fit(X_train, y_train)`, and applies those fitted transformers to test data during `model.predict(X_test)`.

> 💡 **A useful analogy:** the train/test split is like the separation between a study guide and a final exam. You prepare with the study guide (training set) and are evaluated on the final exam (test set) — with no access to final exam questions during studying. Data leakage is like having exam questions in the study guide.


## Part 5: Encoding Categorical Features

Machine learning models require **numeric inputs**. But our Nepal dataset has categorical features: `foundation_type` has values like "Mud mortar-Stone/Brick" and "RC engineered." We need to convert text to numbers.

### Why Numbers?

The sigmoid function and the linear combination require arithmetic: `z = β₀ + β₁·x₁ + ...`. You cannot compute `0.05 × "Mud mortar-Stone/Brick"`. The math only works with numbers.

### Label Encoding / Ordinal Encoding — Why It's Wrong Here

Assign an integer to each category:

```
foundation_type              Encoded
Mud mortar-Stone/Brick  →  0
Bamboo/Timber/Brick     →  1
RC engineered           →  2
Stone/Brick/Wood        →  3
Other                   →  4
```

**The critical problem:** integer encoding implies an **ordering**. It tells the model that Mud (0) < Bamboo (1) < RC (2). But these are **nominal categories** — there is no natural ordering. The model will incorrectly learn that RC (2) is "twice as much foundation" as Bamboo (1).

**When ordinal encoding is correct:** use it for truly ordered categories:
```
Damage Level  →  Encoded
Low           →  0
Medium        →  1
High          →  2
Severe        →  3
```
Here, the order is meaningful: Severe > High > Medium > Low.

> 📌 **Our `foundation_type` values have no meaningful order** — we need One-Hot Encoding.


### One-Hot Encoding (OHE)

Instead of assigning integers, create a **binary column for each category value**:

```
Original feature:
foundation_type
Mud mortar-Stone/Brick
RC engineered
Bamboo/Timber/Brick
...

One-Hot Encoded (5 foundation types → 5 new binary columns):
                 | Mud | Bamboo | RC | Stone | Other |
Mud mortar row   |  1  |   0    |  0 |  0    |   0   |
RC engineered    |  0  |   0    |  1 |  0    |   0   |
Bamboo/Timber    |  0  |   1    |  0 |  0    |   0   |
```

**Why OHE works for nominal categories:**
- No artificial order: each category is represented as an independent binary signal
- The model learns a **separate coefficient** for each category (RC gets β_RC, mud gets β_mud)
- A building that's RC engineered has exactly one `1` in the RC column and `0` in all others — no implicit magnitude comparison

**Feature count expansion:** with our Nepal data:
- `foundation_type`: 5 values → 5 columns
- `roof_type`: several values → several columns
- `ground_floor_type`: several values → several columns
- Total: ~20-30 new binary columns from categorical features

> ⚠️ **The dummy variable trap:** if you include all OHE columns for a feature (say, all 5 foundation type columns), one column is perfectly predictable from the others — `Mud + Bamboo + RC + Stone + Other = 1` always. This creates perfect multicollinearity. Sklearn's `OneHotEncoder` with `drop='first'` or `drop='if_binary'` handles this by dropping one reference category. The `use_cat_names=True` parameter keeps column names interpretable.


## Part 6: Connecting One-Hot Encoding with Logistic Regression

After One-Hot Encoding, every feature is numeric and binary (0 or 1). This is the ideal format for logistic regression.

### The Full Equation After OHE

Before encoding:
```
z = β₀ + β₁·(age) + β₂·(height) + β₃·(foundation_type_TEXT)  ← text: invalid
```

After OHE:
```
z = β₀ + β₁·(age) + β₂·(height)
      + β₃·(foundation_RC) + β₄·(foundation_Mud) + β₅·(foundation_Bamboo)
      + β₆·(roof_RCC) + β₇·(roof_Timber) + ...
```

Now every term is numeric. The logistic regression model learns a separate β coefficient for each OHE column — meaning it learns "being RC foundation → decreases the linear combination by β₃" independently from "being mud mortar foundation → increases it by β₄."

### The sklearn Pipeline

The Pipeline combines the encoder and the classifier into a single unit:

```python
from sklearn.pipeline import Pipeline
from category_encoders import OneHotEncoder
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ('encoder', OneHotEncoder(use_cat_names=True)),
    ('classifier', LogisticRegression(max_iter=1000))
])
model.fit(X_train, y_train)
```

**What Pipeline does:** when you call `model.fit(X_train, y_train)`:
1. Fits the encoder on `X_train` → learns category names from training data only
2. Transforms `X_train` → creates OHE columns
3. Fits logistic regression on the OHE-transformed `X_train`

When you call `model.predict(X_test)`:
1. Transforms `X_test` using the already-fitted encoder (no re-fitting!)
2. Makes predictions using the fitted logistic regression

> 💡 **Why Pipeline prevents leakage:** because the encoder is fitted as part of `model.fit(X_train, ...)`, it never sees test data. The test data is only seen during `model.predict()`, where the encoder transforms (but does not refit).

### Convergence Warnings

You might see: `ConvergenceWarning: lbfgs failed to converge. Increase max_iter.`

**What this means:** logistic regression is solved iteratively. The solver takes small steps toward optimal coefficients. `max_iter=1000` means "take up to 1000 steps." If 1000 steps are not enough to reach the optimum, the solver warns you.

**Effect:** coefficients are slightly suboptimal; accuracy is slightly lower than achievable.

**Fix:** increase to `max_iter=2000`. In our code, `max_iter=1000` is usually sufficient.


## Part 7: Getting Our Data

First, let us import the libraries needed for this lesson:

- **`pandas`**: DataFrame operations, creating pivot tables
- **`numpy`**: numerical operations, exponential for odds ratios
- **`matplotlib.pyplot`**: plotting
- **`seaborn`**: statistical visualization (boxplots, heatmaps)
- **`sklearn`**: Pipeline, train_test_split, LogisticRegression, metrics
- **`category_encoders`**: OneHotEncoder with named columns

**Code 4.2.0.1**: Import required libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    roc_auc_score
)
from sklearn.model_selection import train_test_split
from category_encoders import OneHotEncoder

# Set display options
pd.set_option('display.max_columns', None)

Instead of writing raw SQL queries, we use the `duckdb_wrangle` module created in Lesson 1. This single function call performs the entire data loading pipeline:

> 📋 **What `wrangle_nepal_data()` does automatically:**
> 1. Opens `building_structure.csv` and `building_damage.csv` with DuckDB
> 2. JOINs them with `id_map` on `building_id`
> 3. Filters to `district_id = 4` (Gorkha) by default
> 4. Creates `severe_damage = 1` for Grade 4/5, `0` for Grades 1–3
> 5. Drops `damage_grade` (to prevent target leakage)
> 6. Sets `building_id` as the DataFrame index

The result is a clean DataFrame ready for machine learning — no SQL, no JOIN logic, no manual target encoding.

**Code Task 4.2.1.1**: Import `wrangle_nepal_data` from `duckdb_wrangle` and load the **Gorkha** dataset (district_id=4). Store in `df`.


In [ ]:
from duckdb_wrangle import wrangle_nepal_data

# Load Gorkha district data (district_id = 4)
df = wrangle_nepal_data('./data', district_id=...)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nSevere damage rate: {df['severe_damage'].mean():.2%}")
print(f"\nFirst few rows:")
print(df.head())

---

## Part 8: Exploring the Data

Before building any model, we need to understand our data. Exploratory Data Analysis (EDA) answers four key questions:

1. **Class balance**: are the two classes roughly equal? A 64/36 split means a naïve "always predict majority" classifier gets 64% accuracy — our model must beat this.
2. **Categorical features**: how many unique values does each categorical column have? This tells us how many OHE columns will be created.
3. **Numerical correlations**: are any numerical features highly correlated with each other? (Multicollinearity can destabilize logistic regression.)
4. **Feature–target relationships**: does `foundation_type` or `height` visibly predict severe damage?

**Code Task 4.2.2.1**: Create a bar chart showing the **relative frequency** (not count) of each class in `severe_damage`. Store the normalized value counts in `class_dist`. What does the chart tell you about our baseline accuracy?


In [ ]:
# Calculate class distribution
class_distribution = df['severe_damage'].value_counts(...)

# Create bar plot
fig, ax = plt.subplots(figsize=(9, 6))
class_distribution.plot(kind='bar', ax=ax)
ax.set_xlabel("Severe Damage (0 = No, 1 = Yes)")
ax.set_ylabel("Relative Frequency")
ax.set_title("Distribution of Building Damage")
plt.show()

print(f"Class distribution:\n{class_distribution}")

✅ **You may now attempt Multiple Choice Question 4.2.2.1**

> 📊 **Reading the class distribution:** if `severe_damage = 1` (severe) accounts for ~64% of buildings, this is our **baseline accuracy** — a naïve model that always predicts "severe" would be correct 64% of the time. Our logistic regression must meaningfully exceed 64% to justify its complexity.

---

**Code Task 4.2.2.2**: Check the data types and unique value counts for categorical features. Store the number of unique values for each categorical column in `cat_unique_counts`. This tells us how many OHE columns will be created for each feature.


In [ ]:
# Get categorical columns and their unique value counts
cat_unique_counts = df.select_dtypes(...).nunique()   # <--- 'object'

print("Categorical feature unique value counts:")
print(cat_unique_counts)

---

**Code Task 4.2.2.3**: Create a correlation heatmap of the **numerical features** (excluding the target column `severe_damage`). Store the correlation matrix in `correlation`.

> 🔍 **What to look for in the correlation heatmap:**
> - Correlations close to +1 or −1 indicate features that move together
> - For logistic regression, high multicollinearity (|r| > 0.8) between features can make coefficient estimates unstable
> - If `age_building` and `height_ft_pre_eq` are correlated, that is not surprising — older buildings were often built shorter
> - Features correlated with `severe_damage` would be promising predictors


In [ ]:
import seaborn as sns

# Create correlation matrix for numerical features
correlation = df.select_dtypes('number').drop(columns=...).corr()

# Plot heatmap
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Heatmap of Numerical Features')
plt.show()

print("Correlation matrix created!")
print(correlation)

✅ **You may now attempt Multiple Choice Question 4.2.2.2**

> 📊 **Interpreting the correlation heatmap:** numerical features in our dataset tend to have low correlations with each other, which is good for logistic regression. Features with higher absolute correlation to `severe_damage` will likely contribute more predictive power. Note that categorical features (like `foundation_type`) are NOT shown in the heatmap — their relationships with the target are explored in the pivot table below.

---

**Code 4.2.2.4**: Create a boxplot comparing `height_ft_pre_eq` between severely damaged and non-severely damaged buildings. This visualizes whether building height is a discriminative feature.

> 🔍 **What to look for:** if the median height differs substantially between the two damage classes, height is likely a useful predictor. If the box plots overlap heavily, height alone is not very discriminative — though it may still contribute in combination with other features.


In [ ]:
# Create boxplot
fig, ax = plt.subplots(figsize=(9, 6))
sns.boxplot(x='severe_damage', y='height_ft_pre_eq', data=df, ax=ax)
ax.set_xlabel('Severe Damage (0 = No, 1 = Yes)')
ax.set_ylabel('Height Pre-Earthquake (ft)')
ax.set_title('Building Height by Damage Severity')
plt.show()

print("Boxplot created!")

---

**Code Task 4.2.2.4**: Create a **pivot table** showing the proportion of `severe_damage = 1` for each `foundation_type`. Store in `foundation_damage`. Then create a horizontal bar chart to visualize the damage rate by foundation type.

> 🔍 **What to look for:** which foundation types have the highest severe damage rates? RC (reinforced concrete) engineered foundations are the most resilient building technology — if they show lower severe damage rates, that validates both the data and the physical reasoning. If mud mortar or stone foundations show the highest rates, that is consistent with seismic engineering knowledge.


In [ ]:
# Create pivot table
foundation_damage = pd.pivot_table(
    df, values='...',
    index='foundation_type',
    aggfunc='mean'
).sort_values('...')

print("Severe damage rate by foundation type:")
print(foundation_damage)

# Plot horizontal bar chart
fig, ax = plt.subplots(figsize=(9, 6))
foundation_damage.plot(kind='barh', ax=ax, legend=False)
ax.set_xlabel('Proportion with Severe Damage')
ax.set_title('Severe Damage Rate by Foundation Type')
ax.axvline(x=df['severe_damage'].mean(), color='red', linestyle='--',
           label=f'Overall Average ({df["severe_damage"].mean():.2f})')
ax.legend()
plt.tight_layout()
plt.show()

✅ **You may now attempt Multiple Choice Question 4.2.2.3**

> 📊 **Reading the foundation_type bar chart:** RC engineered foundations should show substantially lower severe damage rates — this is the engineered-to-specification foundation type. Mud mortar and stone/brick foundations were common in older construction and typically perform worst in earthquakes. If the bar chart shows this pattern, it validates that `foundation_type` is a genuinely informative predictor.

---

## Part 9: Preparing for Machine Learning

Now we prepare the data for the modeling pipeline. This involves three sub-steps: (1) define target and features, (2) create the feature matrix and target vector, (3) split into train/test sets.

### Why We Must Split Before Any Transformations

The split happens **before** encoding because the encoder must be fit on training data only. If we encoded first and then split, the encoder would have seen test data — leakage.

**Code Task 4.2.3.1**: Create the target variable name (`target`) and the list of feature column names (`features`). The target is `severe_damage`; the features are all other columns.


In [ ]:
# Define target variable
target = '...'

# Get all columns except target using list comprehension
features = [col for col in df.columns if col != target]

print(f"Target: {target}")
print(f"Number of features: {len(features)}")
print(f"Features: {features}")

---

**Code Task 4.2.3.2**: Create the **feature matrix** `X` and **target vector** `y` from the DataFrame using the `target` and `features` variables.

> 📌 **Naming convention:** `X` is the feature matrix (rows = observations, columns = features). `y` is the target vector (one value per observation). This `(X, y)` convention is universal in sklearn and most ML libraries.


In [ ]:
# Create feature matrix and target vector
X = df[...]
y = df[...]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"\nFirst 5 rows of X:")
print(X.head())
print(f"\nFirst 5 values of y:")
print(y.head())

---

**Code Task 4.2.3.3**: Split the data into training and test sets using `train_test_split`. Use **80% for training**, **20% for testing**, and set `random_state=42` for reproducibility.

> 📌 **What `random_state=42` does:** it seeds the random number generator, ensuring the same split occurs every time you run the code. This is essential for reproducibility — if each run produced a different split, your reported accuracy would fluctuate unpredictably.

> ⚠️ **After this split, never touch `X_test` or `y_test` until final evaluation.** All subsequent steps — including fitting the encoder inside the Pipeline — must use only `X_train` and `y_train`.


In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., random_state=...
)

print(f"Training set: X_train {X_train.shape}, y_train {y_train.shape}")
print(f"Test set: X_test {X_test.shape}, y_test {y_test.shape}")

---

## Part 10: Building the Baseline

Before building our logistic regression model, we need a **baseline accuracy** to compare against. The simplest baseline is the **majority class classifier**: always predict the most frequent class.

### Why Baselines Matter

Suppose our model achieves 68% accuracy on the test set. Is that good?

- If the base rate is 50% (balanced classes): 68% is 18 percentage points above random — meaningful improvement
- If the base rate is 64% (our case): 68% is only 4 percentage points above a naïve classifier — modest improvement
- If the base rate is 90%: 68% would actually be **worse** than always predicting the majority class

**A model is only good if it beats the baseline by a meaningful margin.**

For our dataset: `severe_damage = 1` for ~64% of buildings. A model that always predicts "severe" achieves 64% accuracy without learning anything about building features.

**Code Task 4.2.4.1**: Calculate the baseline accuracy by predicting the majority class (`severe_damage = 1`) for all observations. Store in `baseline_acc`.


In [ ]:
# Calculate baseline accuracy (predict majority class)
baseline_acc = y_train.value_counts(normalize=True)...()  # <--- max()

print(f"Baseline Accuracy: {baseline_acc:.4f}")
print(f"This means predicting '{y_train.value_counts().idxmax()}' for all buildings gives {baseline_acc:.2%} accuracy")

---

## Part 11: Building the Logistic Regression Model

Now we build the full model using a sklearn Pipeline that combines:
1. **OneHotEncoder** (from `category_encoders`): converts categorical text columns to binary columns
2. **LogisticRegression** (from `sklearn.linear_model`): learns coefficients to predict `severe_damage`

### Why Pipeline Instead of Encoding Separately?

```python
# ❌ WRONG (leakage):
encoder = OneHotEncoder().fit(X_train)
X_train_enc = encoder.transform(X_train)
X_test_enc = encoder.transform(X_test)
model = LogisticRegression().fit(X_train_enc, y_train)

# ✓ CORRECT (Pipeline, no leakage):
model = Pipeline([('encoder', OHE), ('clf', LR)]).fit(X_train, y_train)
```

The Pipeline approach is safer and more concise: one `.fit()` call handles everything.

**Code Task 4.2.5.1**: Create a Pipeline with `OneHotEncoder(use_cat_names=True)` as `'encoder'` and `LogisticRegression(max_iter=1000)` as `'classifier'`. Fit it on the training data. Store in `model`.


In [ ]:
from sklearn.pipeline import Pipeline

# Create pipeline with OneHotEncoder and LogisticRegression
model = Pipeline([
    ('encoder', OneHotEncoder(...)),
    ('classifier', LogisticRegression(...))
])

# Fit the model
model.fit(X_train, y_train)

print("Model trained successfully!")
print(f"\nModel steps: {model.steps}")

---

## Part 12: Evaluating the Model

Accuracy alone is insufficient for evaluating a classifier — especially when classes are imbalanced and the costs of different error types are different.

### The Two Error Types

A classification model makes exactly two types of errors:

| Error type | What happened | Nepal consequence |
|-----------|--------------|------------------|
| **False Positive (FP)** | Predicted severe, actually not severe | Inspect a safe building (wasted resources, but safe) |
| **False Negative (FN)** | Predicted not severe, actually severe | Miss a damaged building (no inspection → collapse risk) |

**In disaster response, FN is far more costly than FP.** Missing a severely damaged building could mean families living in a structure at risk of collapse. An extra building inspection wastes time but saves lives.

This asymmetry motivates using **precision** and **recall** alongside accuracy:

| Metric | Formula | What it measures |
|--------|---------|----------------|
| **Accuracy** | (TP + TN) / total | Overall correctness — useful when classes are balanced |
| **Precision** | TP / (TP + FP) | Of all "severe" predictions, how many were actually severe? |
| **Recall** | TP / (TP + FN) | Of all actual severe cases, how many did we catch? |
| **F1 Score** | 2 × P×R / (P+R) | Harmonic mean of precision and recall |

> ⚠️ **In our Nepal context, Recall is the primary metric.** We want to catch as many genuinely severe buildings as possible (high recall), even if it means some false alarms (lower precision).

**Code Task 4.2.6.1**: Calculate training and test accuracy scores. Store in `train_acc` and `test_acc`.


In [ ]:
# Calculate accuracy scores
train_acc = accuracy_score(..., model.predict(...))
test_acc = accuracy_score(..., model.predict(...))

print(f"Training Accuracy: {train_acc:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Baseline Accuracy: {baseline_acc:.4f}")
print(f"\nImprovement over baseline: {test_acc - baseline_acc:.4f}")

---

> 📊 **Reading the accuracy scores:** compare `train_acc` and `test_acc`. A large gap (e.g., 98% train vs 71% test) indicates overfitting — the model memorized training patterns that don't generalize. If both are close to 72-75%, the model is generalizing reasonably well. Both should be above `baseline_acc` (~64%); otherwise the model is not learning anything useful.

**Code Task 4.2.6.2**: Calculate **precision** and **recall** scores for the test set. Store in `precision` and `recall`.

> 📌 **Interpreting precision and recall together:**
> - High precision + low recall: conservative model — only flags high-confidence cases, but misses many actual severe buildings
> - Low precision + high recall: liberal model — catches nearly all severe buildings, but has many false alarms
> - For disaster response: prefer high recall over high precision — missed buildings are more dangerous than false alarms


In [ ]:
# Calculate precision and recall
y_pred = model.predict(X_test)
precision = precision_score(..., ...)
recall = recall_score(..., ...)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print("\nInterpretation:")
print(f"- Precision: When model predicts severe damage, it is correct {precision:.2%} of the time")
print(f"- Recall: Model identifies {recall:.2%} of all severely damaged buildings")

---

### The Confusion Matrix

The **confusion matrix** shows the full breakdown of predictions versus actual labels in a 2×2 table.

```
                  Predicted NOT severe  |  Predicted SEVERE
Actual NOT severe      TN (True Neg)    |    FP (False Pos)  ← Type I error
Actual SEVERE          FN (False Neg)   |    TP (True Pos)   ← Type II error
                       ↑ MISS           |    ↑ HIT
```

**Reading the matrix for Nepal:**
- **TN** (top-left): buildings correctly predicted as NOT severe
- **FP** (top-right): safe buildings incorrectly flagged as severe — wasted inspections
- **FN** (bottom-left): severely damaged buildings MISSED — the dangerous error
- **TP** (bottom-right): severely damaged buildings correctly identified

> ⚠️ **The bottom-left cell (FN) is the one to minimize in disaster response.** A large FN count means many damaged buildings were not flagged for inspection.

**Code 4.2.6.1**: Create and display a confusion matrix. Store the matrix in `cm`.


In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, model.predict(X_test))

# Display confusion matrix
fig, ax = plt.subplots(figsize=(9, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Not Severe', 'Severe'])
disp.plot(ax=ax)
ax.set_title('Confusion Matrix: Earthquake Damage Prediction')
plt.show()

print("Confusion Matrix:")
print(cm)

---

### The ROC Curve and AUC

The **ROC (Receiver Operating Characteristic) curve** shows model performance across *all possible decision thresholds* — not just the default 0.5. This is crucial because in disaster response, we might want to lower the threshold (catch more damaged buildings, accept more false alarms).

**The x and y axes:**
- **x-axis: False Positive Rate (FPR)** = FP / (FP + TN) — what fraction of safe buildings are incorrectly flagged?
- **y-axis: True Positive Rate (TPR = Recall)** = TP / (TP + FN) — what fraction of damaged buildings are caught?

**How to read the ROC curve:**
- **Upper-left corner** (FPR=0, TPR=1): perfect model — catches all damage, no false alarms
- **Diagonal line** (FPR=TPR): random model — equivalent to flipping a coin
- **Our model's curve**: somewhere between random and perfect

**The AUC (Area Under the Curve):** summarizes the entire ROC curve in one number:
- AUC = 1.0: perfect classifier
- AUC = 0.5: random classifier (no better than chance)
- AUC = 0.7: our model can distinguish severe from non-severe buildings ~70% of the time

> 💡 **Why use AUC instead of just accuracy?** AUC is threshold-independent — it measures the model's overall discriminative ability regardless of where you set the decision boundary. It also works well with imbalanced classes where accuracy can be misleading.

**Code 4.2.6.2**: Create and display a ROC curve. Calculate the AUC score.


In [ ]:
# Get predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc_score = roc_auc_score(y_test, y_prob)

# Plot ROC curve
fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(fpr, tpr, label=f'ROC Curve (AUC = {auc_score:.3f})', linewidth=2)
ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)', linewidth=1)
ax.set_xlabel('False Positive Rate (1 - Specificity)')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curve: Logistic Regression')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {auc_score:.4f}")
print(f"\nInterpretation:")
print(f"- AUC = 0.5: Random guessing")
print(f"- AUC = 1.0: Perfect classifier")
print(f"- AUC = {auc_score:.3f}: Our model's performance")

> 📊 **Reading the ROC curve:**
> - The curve should bow toward the upper-left corner — if it hugs the diagonal, the model is barely better than random
> - The AUC score (shown in the legend) tells you the model's overall discriminative ability
> - The "elbow" of the curve (where TPR increases fastest with minimal FPR increase) is often a good choice for the decision threshold in practice
>
> **Adjusting the threshold for disaster response:**
> To increase recall (catch more damaged buildings), lower the threshold below 0.5. For example, at threshold=0.4:
> - Recall will increase (more damaged buildings flagged)
> - Precision will decrease (more safe buildings also flagged)
> - The FPR will move right on the ROC curve
>
> Reading the ROC curve tells you the exact trade-off you're making at each threshold — use it to choose the threshold that matches your operational priorities.

---

## Part 13: Interpreting the Model — Odds Ratios

Let us examine which features are most important for predicting severe damage and in which direction.

**Code Task 4.2.7.1**: Extract feature names and coefficients from the fitted Pipeline. Calculate the **odds ratios** as `exp(coefficient)`. Sort by odds ratio and print the top and bottom 5 features.


In [ ]:
# Extract feature names and coefficients
feature_names = model.named_steps['...'].get_feature_names_out()
coefficients = model.named_steps['...'].coef_[0]

# Calculate odds ratios
odds_ratios = pd.Series(np.exp(coefficients), index=feature_names).sort_values()

print("Top 5 features that DECREASE odds of severe damage:")
print(odds_ratios.head())
print("\nTop 5 features that INCREASE odds of severe damage:")
print(odds_ratios.tail())

In [ ]:
feature_names

> 📊 **Reading the odds ratios:**
>
> **What an odds ratio means:**
> - **Odds ratio = 2.0** for a feature → having that feature doubles the odds of severe damage
> - **Odds ratio = 0.5** for a feature → having that feature halves the odds of severe damage
> - **Odds ratio = 1.0** → no effect (that feature does not change the odds)
>
> **Expected patterns for Nepal earthquake damage:**
> - Foundation type: RC engineered → odds ratio < 1 (reduces severe damage odds); mud mortar → odds ratio > 1 (increases odds)
> - Building age: older buildings → higher odds ratio (more damage)
> - Building height: very tall buildings → higher odds ratio (more collapse risk in earthquakes)
>
> Features with odds ratios far from 1.0 (either much greater or much less than 1) are the most influential predictors.

**Code 4.2.7.1**: Create a horizontal bar chart showing the **top 10 features** with the highest odds ratios (the features that most increase severe damage odds).


In [ ]:
# Plot top 10 odds ratios
fig, ax = plt.subplots(figsize=(9, 6))
odds_ratios.tail(10).plot(kind='barh', ax=ax)
ax.set_xlabel('Odds Ratio')
ax.set_title('Top 10 Features: Odds of Severe Damage')
ax.axvline(x=1, color='red', linestyle='--', label='No Effect (OR=1)')
ax.legend()
plt.tight_layout()
plt.show()

---

## Summary and Discussion

This lesson implemented the full supervised classification pipeline on Nepal earthquake data.

### What You Built and Learned

| Concept | Key takeaway |
|---------|-------------|
| **Binary classification** | Predicts categories (severe vs not severe); model outputs probabilities, not labels directly |
| **Logistic regression** | Linear model that applies the sigmoid to produce calibrated probabilities |
| **Sigmoid function** | Maps any real number z to (0,1); implements the S-curve decision boundary |
| **Train/test split** | Must happen before any transformations; test set is sealed until final evaluation |
| **Data leakage** | Two forms: fitting encoder on full data (Scenario 1); using post-event features (Scenario 2) |
| **One-Hot Encoding** | Converts nominal categories to binary columns; avoids artificial ordering |
| **Pipeline** | Combines encoder + model; automatically prevents leakage |
| **Baseline accuracy** | ~64% (majority class); our model must meaningfully exceed this |
| **Precision** | Of predicted severe cases, how many were actually severe? |
| **Recall** | Of actual severe cases, how many did we catch? Critical metric for disaster response |
| **Confusion matrix** | 2×2 breakdown of TP, TN, FP, FN — FN is the dangerous cell in Nepal context |
| **ROC / AUC** | Performance across all thresholds; AUC = discriminative ability of the model |
| **Odds ratios** | exp(coefficient) — multiplicative effect of each feature on damage odds |

### Key Insights

- Our model achieves **~72% accuracy** on the test set, compared to **~64% baseline** — a meaningful improvement of ~8 percentage points
- **Foundation type** and **superstructure material** are among the strongest predictors
- Buildings with **RC (reinforced concrete) foundations** have substantially lower odds of severe damage — consistent with seismic engineering knowledge
- The model has room for improvement — in Lesson 3 we will try Decision Trees, which can capture non-linear feature interactions

### The Cost Asymmetry Revisited

In disaster response, the FN error (missed damaged building) is more costly than FP (unnecessary inspection). Our baseline logistic regression achieves:
- Some recall improvement over the baseline
- The ROC curve shows the full trade-off space

In L3, we will explore whether Decision Trees can achieve better recall without sacrificing too much precision.

### Discussion Questions

1. **Foundation type interpretation:** why might a building with mud mortar and stone/brick foundation have higher odds of severe damage than an RC engineered foundation? What does this tell you about building material choices in earthquake-prone regions?
2. **Precision vs recall trade-off:** in disaster response, would you prefer higher precision or higher recall? What does your answer imply about the decision threshold you should use (above or below 0.5)?
3. **The baseline comparison:** our model beats the baseline by ~8 percentage points. Is that enough for a disaster response application? What level of accuracy would justify using the model over manual inspection?
4. **Data leakage scenario:** if `condition_post_eq` (building condition after earthquake) were included as a feature, what would happen to train/test accuracy? Why would this be problematic for deployment?
5. **Logistic regression limitations:** the logistic regression decision boundary is linear. What does this mean for buildings whose damage depends on combinations of features (e.g., old age AND mud foundation together produce much higher risk than either alone)? How might Decision Trees handle this differently?
6. **Odds ratio interpretation:** if the odds ratio for `foundation_type_Mud mortar-Stone/Brick` is 1.8, what does this mean for a concrete prediction about a specific building?

### Next Steps

In Lesson 3, you will:
- Build a **Decision Tree** classifier on the same Nepal data
- Learn about **Gini impurity**, **information gain**, and how trees split features
- Explore the **bias-variance trade-off** using `max_depth` as the control knob
- Visualize the decision tree to understand *how* it makes predictions
- Compare Decision Tree performance to Logistic Regression on accuracy, precision, and recall

> ➡️ Lesson 3 starts from the same train/test split and wrangle function. You will see that a Decision Tree can capture non-linear patterns that logistic regression misses — at the cost of higher variance.
